In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [10]:
model = ChatOpenAI(
    model = 'gpt-3.5-turbo',
    max_tokens = 300,
    temperature = 0
)

In [11]:
parser = JsonOutputParser()

In [50]:
'''
template = PromptTemplate(
    template = 'Give me the name, dob and city of 5 freedom figters of India. Exclude Gandhi and Nehru\n {format_instruction}',
    input_variable = [], # We don't have any input variable
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

# format_instruction is a variable but it does not receive value during run time.
# It stores Text which says which type of output we wannt fom the model.
# This text is passed to it using 'partial_variable' which here is 'parser.get_format_instructions()'

prompt = template.invoke(
    {}
)

print(prompt)   # See, text 'Return a JSON object is passed automatically bcz we used 'partial_variable' parameter
'''

template = PromptTemplate(
    template = 'Give me the nameof 5 freedom figters of India and short description on them. Exclude Gandhi and Nehru\n {format_instruction}',
    input_variable = [], # We don't have any input variable
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

# format_instruction is a variable but it does not receive value during run time.
# It stores Text which says which type of output we wannt fom the model.
# This text is passed to it using 'partial_variable' which here is 'parser.get_format_instructions()'

prompt = template.invoke(
    {}
)

print(prompt)   # See, text 'Return a JSON object is passed automatically bcz we used 'partial_variable' parameter


text='Give me the nameof 5 freedom figters of India and short description on them. Exclude Gandhi and Nehru\n Return a JSON object.'


In [25]:
response = model.invoke(prompt)

print(response.content)

{
  "freedom_fighters": [
    {
      "name": "Bhagat Singh",
      "dob": "September 28, 1907",
      "city": "Bangalore"
    },
    {
      "name": "Subhas Chandra Bose",
      "dob": "January 23, 1897",
      "city": "Cuttack"
    },
    {
      "name": "Rani Lakshmibai",
      "dob": "November 19, 1828",
      "city": "Varanasi"
    },
    {
      "name": "Mangal Pandey",
      "dob": "July 19, 1827",
      "city": "Ballia"
    },
    {
      "name": "Chandrashekhar Azad",
      "dob": "July 23, 1906",
      "city": "Jhansi"
    }
  ]
}


In [34]:
final_result = parser.parse(response.content)
print(final_result)
print()
print(type(final_result))   # class dict, bcz python treats json as dictionary

{'freedom_fighters': [{'name': 'Bhagat Singh', 'dob': 'September 28, 1907', 'city': 'Bangalore'}, {'name': 'Subhas Chandra Bose', 'dob': 'January 23, 1897', 'city': 'Cuttack'}, {'name': 'Rani Lakshmibai', 'dob': 'November 19, 1828', 'city': 'Varanasi'}, {'name': 'Mangal Pandey', 'dob': 'July 19, 1827', 'city': 'Ballia'}, {'name': 'Chandrashekhar Azad', 'dob': 'July 23, 1906', 'city': 'Jhansi'}]}

<class 'dict'>


In [46]:
for i in final_result['freedom_fighters']:
    print(i['name'])

Bhagat Singh
Subhas Chandra Bose
Rani Lakshmibai
Mangal Pandey
Chandrashekhar Azad


# Using chain

See we didn't need to write many lines of code. We will always use this approach

In [51]:
chain = template | model | parser

response = chain.invoke({})  # We don't have any input variable in template, so we will have to pass blank dictionary here

print(response)

{'freedom_fighters': [{'name': 'Bhagat Singh', 'description': 'Bhagat Singh was a revolutionary socialist who played a key role in the Indian independence movement. He was known for his acts of defiance against British colonial rule, including the assassination of a British police officer. He was eventually arrested and executed by the British government.'}, {'name': 'Subhas Chandra Bose', 'description': "Subhas Chandra Bose was a prominent leader in the Indian independence movement who believed in using force to overthrow British rule. He formed the Indian National Army to fight against the British during World War II. He is known for his famous slogan 'Give me blood, and I shall give you freedom!'"}, {'name': 'Rani Lakshmibai', 'description': 'Rani Lakshmibai, also known as the Rani of Jhansi, was a queen and warrior who played a significant role in the Indian Rebellion of 1857 against British rule. She led her troops into battle and became a symbol of resistance against colonial opp

# We gets JSON output using JsonOutputParser but it doesn't give us output in any particular schema. Schema is decide by itself.

Suppose the prompt is, `Give the  name of 5 freedom fighters and short description on them` and we want below JSON schema

{'name1': 'description'} <br>
{'name2': 'description'} <br>
{'name3': 'description'} <br>
{'name4': 'description'} <br>

Inshort: `name` becomes key and `description` becomes value

We won't get this. We will have to accept the schema which model returns.

# To get output in our own desired schema, we will see the next part.